# Hafta 11 — Derin Öğrenme Pratiği: MNIST

Bu defter Colab'da **GPU** ile çalıştırılmalı (Runtime → Change runtime type → GPU). İnternet yoksa yedek olarak sklearn'ün 8×8 rakam verisi 28×28'e büyütülerek kullanılır (aynı kod çalışır, doğruluklar farklı olur).

In [ ]:
import numpy as np, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F, time
torch.manual_seed(0); np.random.seed(0)
cihaz = "cuda" if torch.cuda.is_available() else "cpu"; print("cihaz:", cihaz)

def mnist_yukle():
    """Gerçek MNIST (60k+10k, 28×28). İnternet yoksa sklearn digits (1797, 8×8→28×28) ile aynı biçimde döner."""
    try:
        from torchvision import datasets
        tr = datasets.MNIST("./veri", train=True, download=True); te = datasets.MNIST("./veri", train=False, download=True)
        Xtr = tr.data.float().numpy()/255.; ytr = tr.targets.numpy(); Xte = te.data.float().numpy()/255.; yte = te.targets.numpy(); kaynak = "MNIST"
    except Exception as e:
        from sklearn.datasets import load_digits
        from sklearn.model_selection import train_test_split
        d = load_digits(); X = torch.tensor(d.images/16., dtype=torch.float32)[:, None]
        X = F.interpolate(X, size=28, mode="bilinear", align_corners=False)[:, 0].numpy()
        Xtr, Xte, ytr, yte = train_test_split(X, d.target, test_size=0.25, random_state=0, stratify=d.target); kaynak = "sklearn digits (yedek)"
    return Xtr.astype(np.float32), ytr, Xte.astype(np.float32), yte, kaynak

Xtr, ytr, Xte, yte, kaynak = mnist_yukle()
print(kaynak, Xtr.shape, Xte.shape, "sınıf sayısı:", len(np.unique(ytr)))
# Doğrulama kümesi: eğitimin son %10'u
n_val = len(Xtr)//10
Xva, yva = Xtr[-n_val:], ytr[-n_val:]; Xtr, ytr = Xtr[:-n_val], ytr[:-n_val]
T = lambda a, dt=torch.float32: torch.tensor(a, dtype=dt)
Xtr_t, ytr_t, Xva_t, yva_t, Xte_t, yte_t = T(Xtr), T(ytr, torch.long), T(Xva), T(yva, torch.long), T(Xte), T(yte, torch.long)

## 1. Veriye bakalım

In [ ]:
fig, ax = plt.subplots(2, 10, figsize=(10, 2.3))
for i, a in enumerate(ax.ravel()): a.imshow(Xtr[i], cmap="gray"); a.set_title(str(ytr[i]), fontsize=9); a.axis("off")
plt.show(); print("piksel aralığı:", Xtr.min(), Xtr.max(), " sınıf dağılımı:", np.bincount(ytr))

## 2. Softmax + çapraz entropi elle (Örnek 11.1)

In [ ]:
z = np.array([1.0, 2.0, 0.5]); p = np.exp(z)/np.exp(z).sum()
print("p =", p.round(3), " kayıp(y=1) =", round(-np.log(p[1]), 3), " grad =", (p - np.eye(3)[1]).round(3))
zt = torch.tensor(z, requires_grad=True); L = F.cross_entropy(zt[None], torch.tensor([1])); L.backward(); print("PyTorch:", round(L.item(), 3), zt.grad.numpy().round(3))
print("rastgele tahmin kaybı (10 sınıf):", round(np.log(10), 3))

## 3. Eğitim yardımcıları

10. haftanın döngüsü + her epoch'ta eğitim/doğrulama kaydı.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
def dogruluk(model, X, y, bs=1024):
    model.eval(); dogru = 0
    with torch.no_grad():
        for i in range(0, len(X), bs): dogru += (model(X[i:i+bs].to(cihaz)).argmax(1).cpu() == y[i:i+bs]).sum().item()
    return dogru / len(X)

def egit(model, epoch=10, lr=1e-3, bs=128, opt_cls=torch.optim.Adam, wd=0.0, X=None, y=None, sessiz=False):
    X = Xtr_t if X is None else X; y = ytr_t if y is None else y
    model = model.to(cihaz); opt = opt_cls(model.parameters(), lr=lr, weight_decay=wd)
    dl = DataLoader(TensorDataset(X, y), batch_size=bs, shuffle=True); gecmis = {"tr_loss": [], "va_loss": [], "tr_acc": [], "va_acc": []}
    for ep in range(epoch):
        model.train(); toplam = 0
        for xb, yb in dl:
            xb, yb = xb.to(cihaz), yb.to(cihaz); opt.zero_grad(); L = F.cross_entropy(model(xb), yb); L.backward(); opt.step(); toplam += L.item()*len(xb)
        model.eval()
        with torch.no_grad(): va_loss = F.cross_entropy(model(Xva_t.to(cihaz)), yva_t.to(cihaz)).item()
        gecmis["tr_loss"].append(toplam/len(X)); gecmis["va_loss"].append(va_loss); gecmis["tr_acc"].append(dogruluk(model, X, y)); gecmis["va_acc"].append(dogruluk(model, Xva_t, yva_t))
        if not sessiz: print(f"epoch {ep+1:2d}  eğitim kayıp {gecmis['tr_loss'][-1]:.4f} doğruluk {gecmis['tr_acc'][-1]:.4f} | doğrulama kayıp {va_loss:.4f} doğruluk {gecmis['va_acc'][-1]:.4f}")
    return gecmis

def ciz(gecmisler, baslik=""):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    for ad, g in gecmisler.items():
        ax[0].plot(g["tr_loss"], "--", label=f"{ad} eğitim"); ax[0].plot(g["va_loss"], label=f"{ad} doğrulama"); ax[1].plot(g["va_acc"], label=ad)
    ax[0].set_title("kayıp"); ax[1].set_title("doğrulama doğruluğu"); ax[0].legend(fontsize=7); ax[1].legend(fontsize=7)
    for a in ax: a.set_xlabel("epoch"); a.grid(alpha=.3)
    plt.suptitle(baslik); plt.tight_layout(); plt.show()

## 4. Sağlık testi: tek batch'i ezberlet

Kod doğruysa 64 örnekli tek bir batch'te kayıp sıfıra inmeli.

In [ ]:
torch.manual_seed(0); m = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10)).to(cihaz)
xb, yb = Xtr_t[:64].to(cihaz), ytr_t[:64].to(cihaz); o = torch.optim.Adam(m.parameters(), lr=1e-3)
for i in range(200):
    o.zero_grad(); L = F.cross_entropy(m(xb), yb); L.backward(); o.step()
    if i % 50 == 0 or i == 199: print(i, round(L.item(), 4))

## 5. Taban MLP

In [ ]:
torch.manual_seed(0)
mlp = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))
print("parametre:", sum(p.numel() for p in mlp.parameters()))
t0 = time.time(); g_mlp = egit(mlp, epoch=10, lr=1e-3); print(f"süre {time.time()-t0:.1f} s")
ciz({"MLP": g_mlp}); print("test doğruluğu:", round(dogruluk(mlp, Xte_t, yte_t), 4))

## 6. Aşırı öğrenme deneyi: büyük ağ, uzun eğitim, düzenlileştirmesiz vs düzenlileştirmeli

In [ ]:
torch.manual_seed(0)
buyuk = nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 10))
buyuk_reg = nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.ReLU(), nn.Dropout(0.5), nn.Linear(512, 512), nn.ReLU(), nn.Dropout(0.5), nn.Linear(512, 10))
g1 = egit(buyuk, epoch=25, lr=1e-3, sessiz=True); g2 = egit(buyuk_reg, epoch=25, lr=1e-3, wd=1e-4, sessiz=True)
ciz({"düzenlileştirmesiz": g1, "dropout+wd": g2}, "Aşırı öğrenme ve düzenlileştirme")
print("test: düzenlileştirmesiz", round(dogruluk(buyuk, Xte_t, yte_t), 4), " dropout+wd", round(dogruluk(buyuk_reg, Xte_t, yte_t), 4))

**Soru:** Eğitim kaybı hangi modelde daha düşük? Doğrulama kaybı? Hangisini seçersiniz? (İpucu: dropout eğitimde açık olduğu için düzenlileştirilmiş modelin eğitim kaybı yapay olarak yüksek görünür.)

## 7. Erken durdurma (Örnek 11.4)

In [ ]:
def erken_durdurma_ile_egit(model, sabir=3, max_epoch=40, lr=1e-3, wd=0.0):
    model = model.to(cihaz); opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd); dl = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=128, shuffle=True)
    en_iyi, bekleme, en_iyi_ep, va_hist = float("inf"), 0, 0, []
    for ep in range(max_epoch):
        model.train()
        for xb, yb in dl: xb, yb = xb.to(cihaz), yb.to(cihaz); opt.zero_grad(); F.cross_entropy(model(xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad(): va = F.cross_entropy(model(Xva_t.to(cihaz)), yva_t.to(cihaz)).item()
        va_hist.append(va)
        if va < en_iyi: en_iyi, bekleme, en_iyi_ep = va, 0, ep; torch.save(model.state_dict(), "en_iyi.pt")
        else:
            bekleme += 1
            if bekleme >= sabir: print(f"erken durdurma: epoch {ep+1}, en iyi epoch {en_iyi_ep+1} (doğrulama kaybı {en_iyi:.4f})"); break
    model.load_state_dict(torch.load("en_iyi.pt")); return va_hist
torch.manual_seed(0); m = nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 10))
vh = erken_durdurma_ile_egit(m); plt.plot(vh, "o-"); plt.xlabel("epoch"); plt.ylabel("doğrulama kaybı"); plt.grid(alpha=.3); plt.show()
print("test doğruluğu (en iyi epoch ağırlıkları):", round(dogruluk(m, Xte_t, yte_t), 4))

## 8. Öğrenme oranı taraması

In [ ]:
sonuc = {}
for lr in (3e-4, 1e-3, 3e-3, 1e-2):
    torch.manual_seed(0); m = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10)); sonuc[f"lr={lr}"] = egit(m, epoch=8, lr=lr, sessiz=True)
ciz(sonuc, "Öğrenme oranı")

## 9. Batch normalization

In [ ]:
torch.manual_seed(0)
bn = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Linear(128, 10))
duz = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 10))
ciz({"BatchNorm": egit(bn, epoch=8, lr=1e-2, sessiz=True), "düz": egit(duz, epoch=8, lr=1e-2, sessiz=True)}, "Büyük η (1e-2) ile batch norm etkisi")

## 10. Hata analizi: karışıklık matrisi ve yanlış örnekler

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
buyuk_reg.eval()
with torch.no_grad(): pred = buyuk_reg(Xte_t.to(cihaz)).argmax(1).cpu().numpy()
ConfusionMatrixDisplay.from_predictions(yte, pred, cmap="Blues"); plt.show()
yanlis = np.where(pred != yte)[0]; print("yanlış sayısı:", len(yanlis), "/", len(yte))
fig, ax = plt.subplots(2, 10, figsize=(12, 3))
for a, i in zip(ax.ravel(), yanlis[:20]): a.imshow(Xte[i], cmap="gray"); a.set_title(f"g{yte[i]} t{pred[i]}", fontsize=8); a.axis("off")
plt.show()

**Soru:** En çok karışan rakam çifti hangisi? Yanlış örneklerin kaçı sizce insan için de belirsiz?

## 11. Alıştırmalar

**Alıştırma 1.** z = [2, 1, 0] ve y = 0 için softmax, kayıp ve gradyanı elle hesaplayıp kodla doğrulayın. Aynı z için y = 2 kaybını da hesaplayın.

In [ ]:
# Alıştırma 1

**Alıştırma 2.** Dropout oranını p ∈ {0, 0.2, 0.5, 0.8} için tarayın (512-512 ağ, 15 epoch). Doğrulama doğruluğunu p'ye karşı çizin. Çok büyük p ne yapıyor?

In [ ]:
# Alıştırma 2

**Alıştırma 3.** Modelin son katmanına yanlışlıkla `nn.Softmax(dim=1)` ekleyip `CrossEntropyLoss` ile eğitin. Kayıp ve doğruluk ne oluyor, neden? (İpucu: çift softmax gradyanı sıkıştırır.)

In [ ]:
# Alıştırma 3

**Alıştırma 4.** Veri artırma: eğitim görüntülerini her batch'te rastgele ±2 piksel kaydırın (`torch.roll`) ve ±10° döndürün (`torchvision.transforms.functional.rotate`). 25 epoch'ta artırmalı ve artırmasız doğrulama/test doğruluğunu karşılaştırın.

In [ ]:
# Alıştırma 4

**Alıştırma 5.** Eğitim kümesinin yalnızca %5'i ile (rastgele) aynı MLP'yi eğitin. Eğitim/doğrulama farkı nasıl değişti? Düzenlileştirme bu durumda ne kadar yardımcı oluyor? Veri miktarı ile aşırı öğrenme ilişkisini bir cümleyle yazın.

In [ ]:
# Alıştırma 5